<a href="https://colab.research.google.com/github/darioLabrador/TestRepo/blob/main/(FO)Training_Colour.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!pip install torch
!pip install numpy
!pip install open_clip-torch
!pip install Pillow
!pip install scikit-learn
!pip install torch pandas

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 79.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 4.4 MB/s eta 0:00:00


In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
import torch
import numpy as np
import random
import open_clip
from PIL import Image
import pandas as pd
import glob

def set_seed(seed=42):
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    np.random.seed(seed)
    random.seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(42)

# Setup Model & Data
model_id = 'hf-hub:microsoft/BiomedCLIP-PubMedBERT_256-vit_base_patch16_224'
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Loading BiomedCLIP model on {device}...")

model, _, preprocess_val = open_clip.create_model_and_transforms(model_id)
tokenizer = open_clip.get_tokenizer(model_id)
model.eval()
model.to(device)

# Automatically fetch all .png files from the directory
image_paths = np.array(sorted(glob.glob("/content/drive/MyDrive/Test Data(37)/high_risk_subset/*.png")))
image_paths2 = np.array(sorted(glob.glob("/content/drive/MyDrive/Test Data(37)/no_risk_subset/*.png")))

# Define Label Sets & Pre-encode Colour
colour_labels = [
    "a T wave with a high amount of blue/green",
    "a T wave with a high amount of orange/red",
]

# Pre-encode colour features globally so evaluate_image can use them
text_c = tokenizer(colour_labels).to(device)
with torch.no_grad():
    text_features_c = model.encode_text(text_c)
    text_features_c /= text_features_c.norm(dim=-1, keepdim=True)

morphology_label_sets = {
    "No Label": [
        "Normal",
        "Abnormal"
    ],
    "L1": [
        "Normal T wave",
        "Abnormal T wave"
    ],
    "L2": [
        "Normal",
        "Flat",
        "Large",
        "Inverted",
        "Biphasic",
        "Notched"
    ],
    "L3": [
        "Normal T wave",
        "Abnormal T wave with flat amplitude",
        "Abnormal large T wave, taller than initial signal",
        """Abnormal large T wave, broader and larger than
         initial signal""",
        "Abnormal T wave with a minor inversion",
        "Abnormal T wave with a deep inversion",
        "Abnormal T wave with a giant inversion",
        "Abnormal Biphasic T wave(Type A)",
        "Abnormal Biphasic T wave(Type B)",
        "Abnormal Bifid or Notched T wave"
    ]
}

# Helper Function for Inference
def evaluate_image(image_path, text_features_m, text_features_c, morphology_labels, alpha=0.5):
    """Returns probability arrays for Morphology, Colour, Both, and None"""
    raw_image = Image.open(image_path).convert('RGB')
    image = preprocess_val(raw_image).unsqueeze(0).to(device)

    with torch.no_grad():
        image_features = model.encode_image(image)
        image_features /= image_features.norm(dim=-1, keepdim=True)
        logit_scale = model.logit_scale.exp()

        # Logits and basic Probs
        logits_per_image_m = logit_scale * image_features @ text_features_m.t()
        logits_per_image_c = logit_scale * image_features @ text_features_c.t()

        probs_m = logits_per_image_m.softmax(dim=-1).cpu().numpy()[0]
        probs_c = logits_per_image_c.softmax(dim=-1).cpu().numpy()[0]

        # Fusion (Both)
        p_blue_green = probs_c[0]
        p_orange_red = probs_c[1]

        weight_blue_green = (p_blue_green * alpha) + (1.0 - alpha)
        weight_orange_red = (p_orange_red * alpha) + (1.0 - alpha)

        mixed_probs_m = []
        for i, label in enumerate(morphology_labels):
            weight = weight_blue_green if "Normal" in label else weight_orange_red
            mixed_probs_m.append(probs_m[i] * weight)

        mixed_probs_m = np.array(mixed_probs_m)
        mixed_probs_m /= mixed_probs_m.sum()

        # NONE (Baseline uniform distribution)
        none_probs = np.ones(len(morphology_labels)) / len(morphology_labels)

    return probs_m, probs_c, mixed_probs_m, none_probs


Loading BiomedCLIP model on cuda...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


open_clip_config.json:   0%|          | 0.00/707 [00:00<?, ?B/s]

open_clip_pytorch_model.bin:   0%|          | 0.00/784M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/385 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/28.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

In [5]:
import glob
from PIL import Image
import numpy as np
import torch
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

print("Loading training data...")

# Define training paths
train_high_risk_dir = "/content/drive/MyDrive/Training Data(144)/high_risk_subset/*.png"
train_no_risk_dir = "/content/drive/MyDrive/Training Data(144)/no_risk_subset/*.png"

train_paths_abnormal = sorted(glob.glob(train_high_risk_dir))
train_paths_normal = sorted(glob.glob(train_no_risk_dir))

train_paths = train_paths_abnormal + train_paths_normal
# Labels: 1 for Abnormal, 0 for Normal
train_labels = [1] * len(train_paths_abnormal) + [0] * len(train_paths_normal)

print(f"Found {len(train_paths_abnormal)} Abnormal and {len(train_paths_normal)} Normal training images.")

def extract_features(paths):
    features = []
    model.eval()
    with torch.no_grad():
        for path in paths:
            try:
                raw_image = Image.open(path).convert('RGB')
                image = preprocess_val(raw_image).unsqueeze(0).to(device)
                image_features = model.encode_image(image)
                image_features /= image_features.norm(dim=-1, keepdim=True)
                features.append(image_features.cpu().numpy()[0])
            except Exception as e:
                print(f"Error processing {path}: {e}")
                # Append a zero vector if image fails to load to maintain alignment, though ideally it should be removed
                features.append(np.zeros(model.visual.output_dim))
    return np.array(features)

print("Extracting embeddings using BiomedCLIP... (This may take a moment)")
X_train = extract_features(train_paths)
y_train = np.array(train_labels)

# Train a simple classifier
print("Training Logistic Regression classifier...")
classifier = LogisticRegression(random_state=42, max_iter=1000)
classifier.fit(X_train, y_train)

# Evaluate on training data to verify it learned
train_preds = classifier.predict(X_train)
print("\nTraining Classification Report:")
print(classification_report(y_train, train_preds, target_names=['Normal', 'Abnormal']))

Loading training data...
Found 144 Abnormal and 144 Normal training images.
Extracting embeddings using BiomedCLIP... (This may take a moment)
Training Logistic Regression classifier...

Training Classification Report:
              precision    recall  f1-score   support

      Normal       0.96      0.99      0.98       144
    Abnormal       0.99      0.96      0.98       144

    accuracy                           0.98       288
   macro avg       0.98      0.98      0.98       288
weighted avg       0.98      0.98      0.98       288



In [6]:
import glob
import numpy as np
from sklearn.metrics import classification_report

print("Extracting embeddings for the Test Data(37) using BiomedCLIP...")
# Ensure we grab both abnormal and normal test images
test_high_risk_dir = "/content/drive/MyDrive/Test Data(37)/high_risk_subset/*.png"
test_no_risk_dir = "/content/drive/MyDrive/Test Data(37)/no_risk_subset/*.png"

test_paths_abnormal = sorted(glob.glob(test_high_risk_dir))
test_paths_normal = sorted(glob.glob(test_no_risk_dir))

all_test_paths = test_paths_abnormal + test_paths_normal

# Convert to binary (1 for Abnormal, 0 for Normal) to match training
y_test_binary = np.array([1] * len(test_paths_abnormal) + [0] * len(test_paths_normal))

X_test = extract_features(all_test_paths)

print("Evaluating Logistic Regression classifier on Test Data(37)...")
test_preds = classifier.predict(X_test)

print("\nTest Classification Report (Linear Probe):")
print(classification_report(y_test_binary, test_preds, target_names=['Normal', 'Abnormal']))

Extracting embeddings for the Test Data(37) using BiomedCLIP...
Evaluating Logistic Regression classifier on Test Data(37)...

Test Classification Report (Linear Probe):
              precision    recall  f1-score   support

      Normal       1.00      0.35      0.52        37
    Abnormal       0.61      1.00      0.76        37

    accuracy                           0.68        74
   macro avg       0.80      0.68      0.64        74
weighted avg       0.80      0.68      0.64        74



In [7]:
import glob
import numpy as np

# Define real image paths
high_risk_dir = "/content/drive/MyDrive/Test Data(37)/high_risk_subset/*.png"
no_risk_dir = "/content/drive/MyDrive/Test Data(37)/no_risk_subset/*.png"

# Retrieve and sort .png files
image_paths_abnormal = np.array(sorted(glob.glob(high_risk_dir)))
image_paths_normal = np.array(sorted(glob.glob(no_risk_dir)))

# Create ground truth labels
ground_truth_abnormal = np.array(['Abnormal'] * len(image_paths_abnormal))
ground_truth_normal = np.array(['Normal'] * len(image_paths_normal))

# Combine arrays
image_paths = np.concatenate((image_paths_abnormal, image_paths_normal))
ground_truth = np.concatenate((ground_truth_abnormal, ground_truth_normal))

# Print counts
print(f"Found {len(image_paths_abnormal)} high-risk (abnormal) images.")
print(f"Found {len(image_paths_normal)} no-risk (normal) images.")
print(f"Total combined images: {len(image_paths)}")
print(f"Total ground truth labels: {len(ground_truth)}")

Found 37 high-risk (abnormal) images.
Found 37 no-risk (normal) images.
Total combined images: 74
Total ground truth labels: 74


In [8]:
import glob
import numpy as np

# Define real image paths
high_risk_dir = "/content/drive/MyDrive/HeartBeat_Coloured/All/At risk/*.png"
no_risk_dir = "/content/drive/MyDrive/HeartBeat_Coloured/All/No risk/*.png"

# Retrieve and sort .png files
image_paths_abnormal = np.array(sorted(glob.glob(high_risk_dir)))
image_paths_normal = np.array(sorted(glob.glob(no_risk_dir)))

# Create ground truth labels
ground_truth_abnormal = np.array(['Abnormal'] * len(image_paths_abnormal))
ground_truth_normal = np.array(['Normal'] * len(image_paths_normal))

# Combine arrays
image_paths = np.concatenate((image_paths_abnormal, image_paths_normal))
ground_truth = np.concatenate((ground_truth_abnormal, ground_truth_normal))

# Print counts
print(f"Found {len(image_paths_abnormal)} high-risk (abnormal) images.")
print(f"Found {len(image_paths_normal)} no-risk (normal) images.")
print(f"Total combined images: {len(image_paths)}")
print(f"Total ground truth labels: {len(ground_truth)}")


Found 181 high-risk (abnormal) images.
Found 4854 no-risk (normal) images.
Total combined images: 5035
Total ground truth labels: 5035


In [9]:
import pandas as pd
import numpy as np
import torch
import os

real_inference_results_colour = []
desired_label_set_order = ["No Label", "L1", "L2", "L3"]

print("Starting inference loop including Colour Strategy...")
for label_set_name in desired_label_set_order:
    morph_labels = morphology_label_sets[label_set_name]

    # Pre-encode text features for the current morphology labels
    text_m = tokenizer(morph_labels).to(device)
    with torch.no_grad():
        text_features_m = model.encode_text(text_m)
        text_features_m /= text_features_m.norm(dim=-1, keepdim=True)

    # Iterate through all real image paths
    for path, true_label in zip(image_paths, ground_truth):
        try:
            # Evaluate image
            p_morph, p_col, p_both, p_none = evaluate_image(
                path, text_features_m, text_features_c, morph_labels, alpha=0.5
            )

            # Extract predicted labels
            pred_morph = morph_labels[np.argmax(p_morph)]
            pred_both = morph_labels[np.argmax(p_both)]
            pred_none = morph_labels[np.argmax(p_none)]

            # Colour Strategy mapping: blue/green (index 0) -> Normal, orange/red (index 1) -> Abnormal
            pred_col = 'Normal' if np.argmax(p_col) == 0 else 'Abnormal'

            # Extract file name from path
            file_name = os.path.basename(path)

            # Morphology Strategy
            real_inference_results_colour.append({
                'File_Name': file_name,
                'True_Label': true_label,
                'Predicted_Label': pred_morph,
                'Strategy': 'No Colour + Morphology Context',
                'Label_Set': label_set_name
            })

            # Mixed Strategy
            real_inference_results_colour.append({
                'File_Name': file_name,
                'True_Label': true_label,
                'Predicted_Label': pred_both,
                'Strategy': 'Colour + Morphology Context',
                'Label_Set': label_set_name
            })

            # Random Strategy
            real_inference_results_colour.append({
                'File_Name': file_name,
                'True_Label': true_label,
                'Predicted_Label': pred_none,
                'Strategy': 'No Colour + No Morphology Context',
                'Label_Set': label_set_name
            })

            # Colour Strategy
            real_inference_results_colour.append({
                'File_Name': file_name,
                'True_Label': true_label,
                'Predicted_Label': pred_col,
                'Strategy': 'Colour + No Morphology Context',
                'Label_Set': label_set_name
            })

        except Exception as e:
            print(f"Error processing {path}: {e}")

# Convert results to DataFrame
df_real_inference_colour = pd.DataFrame(real_inference_results_colour)
print("\nInference including Colour Strategy complete! First few results:")
display(df_real_inference_colour.head(10))

Starting inference loop including Colour Strategy...

Inference including Colour Strategy complete! First few results:


,File_Name,True_Label,Predicted_Label,Strategy,Label_Set
0,1415_Heartbeat_1.png,Abnormal,Abnormal,No Colour + Morphology Context,No Label
1,1415_Heartbeat_1.png,Abnormal,Abnormal,Colour + Morphology Context,No Label
2,1415_Heartbeat_1.png,Abnormal,Normal,No Colour + No Morphology Context,No Label
3,1415_Heartbeat_1.png,Abnormal,Abnormal,Colour + No Morphology Context,No Label
4,1426_Heartbeat_1.png,Abnormal,Abnormal,No Colour + Morphology Context,No Label
5,1426_Heartbeat_1.png,Abnormal,Abnormal,Colour + Morphology Context,No Label
6,1426_Heartbeat_1.png,Abnormal,Normal,No Colour + No Morphology Context,No Label
7,1426_Heartbeat_1.png,Abnormal,Abnormal,Colour + No Morphology Context,No Label
8,1441_Heartbeat_1.png,Abnormal,Abnormal,No Colour + Morphology Context,No Label
9,1441_Heartbeat_1.png,Abnormal,Abnormal,Colour + Morphology Context,No Label


In [10]:
import pandas as pd
import numpy as np
import math
from sklearn.metrics import roc_curve, roc_auc_score, confusion_matrix
import matplotlib.pyplot as plt

# Map detailed predictions to binary
def map_to_binary(label):
    if 'normal' in label.lower() and 'abnormal' not in label.lower():
        return 'Normal'
    elif 'normal' in label.lower() and label.lower().startswith('normal'):
         return 'Normal'
    else:
        return 'Abnormal'

df_real_inference_colour['Binary_Prediction'] = df_real_inference_colour['Predicted_Label'].apply(map_to_binary)

# Map to numeric binary for sklearn metrics
df_real_inference_colour['True_Binary_Num'] = (df_real_inference_colour['True_Label'] == 'Abnormal').astype(int)
df_real_inference_colour['Pred_Binary_Num'] = (df_real_inference_colour['Binary_Prediction'] == 'Abnormal').astype(int)

# Initialize metrics list
metrics_colour_results = []

# Group and calculate metrics
grouped_colour = df_real_inference_colour.groupby(['Label_Set', 'Strategy'])

plt.figure(figsize=(12, 8))

for (label_set, strategy), group in grouped_colour:
    y_true = group['True_Binary_Num']
    y_pred = group['Pred_Binary_Num']

    TP = len(group[(group['True_Label'] == 'Abnormal') & (group['Binary_Prediction'] == 'Abnormal')])
    FN = len(group[(group['True_Label'] == 'Abnormal') & (group['Binary_Prediction'] == 'Normal')])
    TN = len(group[(group['True_Label'] == 'Normal') & (group['Binary_Prediction'] == 'Normal')])
    FP = len(group[(group['True_Label'] == 'Normal') & (group['Binary_Prediction'] == 'Abnormal')])

    recall = TP / (TP + FN) if (TP + FN) > 0 else 0.0
    specificity = TN / (TN + FP) if (TN + FP) > 0 else 0.0

    # New Metrics
    ppv = TP / (TP + FP) if (TP + FP) > 0 else 0.0
    f1 = 2 * (ppv * recall) / (ppv + recall) if (ppv + recall) > 0 else 0.0

    denom = math.sqrt((TP + FP) * (TP + FN) * (TN + FP) * (TN + FN))
    mcc = ((TP * TN) - (FP * FN)) / denom if denom > 0 else 0.0

    # ROC AUC
    try:
        roc_auc = roc_auc_score(y_true, y_pred)
    except ValueError:
        roc_auc = 0.0

    # Append results
    metrics_colour_results.append({
        'Label_Set': label_set,
        'Strategy': strategy,
        'TP': TP,
        'FN': FN,
        'TN': TN,
        'FP': FP,
        'Sensitivity': recall,
        'Specificity': specificity,
        'PPV': ppv,
        'F1_Score': f1,
        'MCC': mcc,
        'ROC_AUC': roc_auc
    })

# Linear Probe
if 'y_test_binary' in globals() and 'test_preds' in globals():
    tn, fp, fn, tp = confusion_matrix(y_test_binary, test_preds).ravel()
    recall_lp = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    specificity_lp = tn / (tn + fp) if (tn + fp) > 0 else 0.0
    ppv_lp = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    f1_lp = 2 * (ppv_lp * recall_lp) / (ppv_lp + recall_lp) if (ppv_lp + recall_lp) > 0 else 0.0

    denom_lp = math.sqrt((tp + fp) * (tp + fn) * (tn + fp) * (tn + fn))
    mcc_lp = ((tp * tn) - (fp * fn)) / denom_lp if denom_lp > 0 else 0.0
    roc_auc_lp = roc_auc_score(y_test_binary, test_preds)

    fpr_lp, tpr_lp, _ = roc_curve(y_test_binary, test_preds)

    metrics_colour_results.append({
        'Label_Set': 'N/A (Trained)',
        'Strategy': 'Linear Probe',
        'TP': tp,
        'FN': fn,
        'TN': tn,
        'FP': fp,
        'Sensitivity': recall_lp,
        'Specificity': specificity_lp,
        'PPV': ppv_lp,
        'F1_Score': f1_lp,
        'MCC': mcc_lp,
        'ROC_AUC': roc_auc_lp
    })

# Output Dataframe
df_metrics_colour = pd.DataFrame(metrics_colour_results)
print("Final Metrics Including Colour Strategy, Linear Probe, and Additional Metrics:")
display(df_metrics_colour.sort_values(by='ROC_AUC', ascending=False))

Final Metrics Including Colour Strategy, Linear Probe, and Additional Metrics:


,Label_Set,Strategy,TP,FN,TN,FP,Sensitivity,Specificity,PPV,F1_Score,MCC,ROC_AUC
16,N/A (Trained),Linear Probe,37,0,13,24,1.000000,0.351351,0.606557,0.755102,0.461644,0.675676
13,No Label,Colour + No Morphology Context,175,6,972,3882,0.966851,0.200247,0.043135,0.082586,0.078630,0.583549
1,L1,Colour + No Morphology Context,175,6,972,3882,0.966851,0.200247,0.043135,0.082586,0.078630,0.583549
9,L3,Colour + No Morphology Context,175,6,972,3882,0.966851,0.200247,0.043135,0.082586,0.078630,0.583549
5,L2,Colour + No Morphology Context,175,6,972,3882,0.966851,0.200247,0.043135,0.082586,0.078630,0.583549
0,L1,Colour + Morphology Context,97,84,3028,1826,0.535912,0.623815,0.050442,0.092205,0.061201,0.579864
2,L1,No Colour + Morphology Context,56,125,3889,965,0.309392,0.801195,0.054848,0.093178,0.051203,0.555294
8,L3,Colour + Morphology Context,47,134,4114,740,0.259669,0.847548,0.059720,0.097107,0.054963,0.553608
12,No Label,Colour + Morphology Context,159,22,850,4004,0.878453,0.175113,0.038194,0.073204,0.026352,0.526783
10,L3,No Colour + Morphology Context,24,157,4408,446,0.132597,0.908117,0.051064,0.073733,0.026053,0.520357


<Figure size 1200x800 with 0 Axes>

In [11]:
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, confusion_matrix
import torch
from PIL import Image
import math

# Prepare binary labels for real data (1 = Abnormal, 0 = Normal)
y_true_numeric = np.array([1 if t == 'Abnormal' else 0 for t in ground_truth])

print("Pre-extracting features for all real images...")
# Pre-extract image embeddings and zero-shot probabilities
image_embeddings_list = []
p_morph_dict = {ls: [] for ls in desired_label_set_order}
p_col_dict = {ls: [] for ls in desired_label_set_order}
p_both_dict = {ls: [] for ls in desired_label_set_order}
p_none_dict = {ls: [] for ls in desired_label_set_order}

model.eval()
with torch.no_grad():
    for path in image_paths:
        try:
            raw_image = Image.open(path).convert('RGB')
            image = preprocess_val(raw_image).unsqueeze(0).to(device)
            img_feat = model.encode_image(image)
            img_feat /= img_feat.norm(dim=-1, keepdim=True)
            img_feat_np = img_feat.cpu().numpy()[0]
        except Exception as e:
            print(f"Error loading {path}: {e}")
            img_feat_np = np.zeros(model.visual.output_dim)

        image_embeddings_list.append(img_feat_np)

        # Get text probabilities for each label set
        for label_set_name in desired_label_set_order:
            morph_labels = morphology_label_sets[label_set_name]
            text_m = tokenizer(morph_labels).to(device)
            text_features_m = model.encode_text(text_m)
            text_features_m /= text_features_m.norm(dim=-1, keepdim=True)

            p_morph, p_col, p_both, p_none = evaluate_image(
                path, text_features_m, text_features_c, morph_labels, alpha=0.5
            )

            p_morph_dict[label_set_name].append(p_morph)
            p_col_dict[label_set_name].append(p_col)
            p_both_dict[label_set_name].append(p_both)
            p_none_dict[label_set_name].append(p_none)

X_emb = np.array(image_embeddings_list)

print("Extraction complete. Running 5-Fold Cross Validation...")
# Run Cross-Validation for each strategy and label set
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
concat_cv_results = []

# Tracking individual predictions for distribution analysis
cv_predictions_list = []

for label_set_name in desired_label_set_order:
    features_strategies = {
        'No Colour + Morphology': np.array(p_morph_dict[label_set_name]),
        'Colour + No Morphology': np.array(p_col_dict[label_set_name]),
        'Colour + Morphology': np.array(p_both_dict[label_set_name]),
        'No Colour + No Morphology': np.array(p_none_dict[label_set_name])
    }

    for strategy_name, strategy_probs in features_strategies.items():
        # Concatenate image embeddings with strategy probabilities
        X_concat = np.hstack((X_emb, strategy_probs))

        fold_y_true = []
        fold_y_pred = []
        fold_y_prob = []

        for train_idx, val_idx in skf.split(X_concat, y_true_numeric):
            X_train_cv, X_val_cv = X_concat[train_idx], X_concat[val_idx]
            y_train_cv, y_val_cv = y_true_numeric[train_idx], y_true_numeric[val_idx]

            clf = LogisticRegression(max_iter=1000, random_state=42)
            clf.fit(X_train_cv, y_train_cv)

            preds = clf.predict(X_val_cv)
            probs = clf.predict_proba(X_val_cv)[:, 1]

            fold_y_true.extend(y_val_cv)
            fold_y_pred.extend(preds)
            fold_y_prob.extend(probs)

        # Track predictions for distribution analysis
        for p in fold_y_pred:
            cv_predictions_list.append({
                'Label_Set': label_set_name,
                'Strategy': strategy_name,
                'Binary_Prediction': 'Abnormal' if p == 1 else 'Normal'
            })

        # Calculate metrics over all out-of-fold predictions
        tn, fp, fn, tp = confusion_matrix(fold_y_true, fold_y_pred).ravel()
        recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
        specificity = tn / (tn + fp) if (tn + fp) > 0 else 0.0
        auc = roc_auc_score(fold_y_true, fold_y_prob)

        ppv = tp / (tp + fp) if (tp + fp) > 0 else 0.0
        f1 = 2 * (ppv * recall) / (ppv + recall) if (ppv + recall) > 0 else 0.0
        denom = math.sqrt((tp + fp) * (tp + fn) * (tn + fp) * (tn + fn))
        mcc = ((tp * tn) - (fp * fn)) / denom if denom > 0 else 0.0

        concat_cv_results.append({
            'Label_Set': label_set_name,
            'Strategy': strategy_name,
            'TP': tp,
            'FN': fn,
            'TN': tn,
            'FP': fp,
            'Sensitivity (Recall)': recall,
            'Specificity': specificity,
            'PPV': ppv,
            'F1_Score': f1,
            'MCC': mcc,
            'ROC_AUC': auc
        })

df_concat_cv = pd.DataFrame(concat_cv_results)

display(df_concat_cv)

Pre-extracting features for all real images...
Extraction complete. Running 5-Fold Cross Validation...


,Label_Set,Strategy,TP,FN,TN,FP,Sensitivity (Recall),Specificity,PPV,F1_Score,MCC,ROC_AUC
0,No Label,No Colour + Morphology,0,181,4854,0,0.0,1.0,0.0,0.0,0.0,0.932031
1,No Label,Colour + No Morphology,0,181,4854,0,0.0,1.0,0.0,0.0,0.0,0.922853
2,No Label,Colour + Morphology,0,181,4854,0,0.0,1.0,0.0,0.0,0.0,0.930930
3,No Label,No Colour + No Morphology,0,181,4854,0,0.0,1.0,0.0,0.0,0.0,0.936519
4,L1,No Colour + Morphology,0,181,4854,0,0.0,1.0,0.0,0.0,0.0,0.928369
5,L1,Colour + No Morphology,0,181,4854,0,0.0,1.0,0.0,0.0,0.0,0.922853
6,L1,Colour + Morphology,0,181,4854,0,0.0,1.0,0.0,0.0,0.0,0.917999
7,L1,No Colour + No Morphology,0,181,4854,0,0.0,1.0,0.0,0.0,0.0,0.936519
8,L2,No Colour + Morphology,0,181,4854,0,0.0,1.0,0.0,0.0,0.0,0.918502
9,L2,Colour + No Morphology,0,181,4854,0,0.0,1.0,0.0,0.0,0.0,0.922853


In [12]:
from sklearn.model_selection import StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, confusion_matrix
import pandas as pd
import numpy as np
import math

print("Running 5-Fold Cross Validation to output per-fold metrics...")
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
fold_cv_results = []

for label_set_name in desired_label_set_order:
    features_strategies = {
        'No Colour + Morphology': np.array(p_morph_dict[label_set_name]),
        'Colour + No Morphology': np.array(p_col_dict[label_set_name]),
        'Colour + Morphology': np.array(p_both_dict[label_set_name]),
        'No Colour + No Morphology': np.array(p_none_dict[label_set_name])
    }

    for strategy_name, strategy_probs in features_strategies.items():
        # Concatenate image embeddings with strategy probabilities
        X_concat = np.hstack((X_emb, strategy_probs))

        fold_idx = 1
        for train_idx, val_idx in skf.split(X_concat, y_true_numeric):
            X_train_cv, X_val_cv = X_concat[train_idx], X_concat[val_idx]
            y_train_cv, y_val_cv = y_true_numeric[train_idx], y_true_numeric[val_idx]

            # Apply class_weight='balanced'
            clf = LogisticRegression(max_iter=1000, random_state=42, class_weight='balanced')
            clf.fit(X_train_cv, y_train_cv)

            preds = clf.predict(X_val_cv)
            probs = clf.predict_proba(X_val_cv)[:, 1]

            # Calculate metrics for the current fold
            tn, fp, fn, tp = confusion_matrix(y_val_cv, preds).ravel()
            recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
            specificity = tn / (tn + fp) if (tn + fp) > 0 else 0.0

            try:
                auc = roc_auc_score(y_val_cv, probs)
            except ValueError:
                auc = 0.0

            ppv = tp / (tp + fp) if (tp + fp) > 0 else 0.0
            f1 = 2 * (ppv * recall) / (ppv + recall) if (ppv + recall) > 0 else 0.0
            denom = math.sqrt((tp + fp) * (tp + fn) * (tn + fp) * (tn + fn))
            mcc = ((tp * tn) - (fp * fn)) / denom if denom > 0 else 0.0

            fold_cv_results.append({
                'Label_Set': label_set_name,
                'Strategy': strategy_name,
                'Fold': fold_idx,
                'TP': tp,
                'FN': fn,
                'TN': tn,
                'FP': fp,
                'Sensitivity': recall,
                'Specificity': specificity,
                'PPV': ppv,
                'F1_Score': f1,
                'MCC': mcc,
                'ROC_AUC': auc
            })
            fold_idx += 1

df_fold_cv = pd.DataFrame(fold_cv_results)

# Desired columns to display
display_columns = [
    'Label_Set', 'Strategy', 'TP', 'FN', 'TN', 'FP',
    'Sensitivity', 'Specificity', 'PPV', 'F1_Score', 'MCC', 'ROC_AUC'
]

# Display a separate table for each fold
for fold in range(1, 6):
    print(f"\n{'='*60}")
    print(f"Metrics for Fold {fold}")
    print(f"{'='*60}")
    df_fold = df_fold_cv[df_fold_cv['Fold'] == fold][display_columns].copy()
    display(df_fold)


Running 5-Fold Cross Validation to output per-fold metrics...

Metrics for Fold 1


,Label_Set,Strategy,TP,FN,TN,FP,Sensitivity,Specificity,PPV,F1_Score,MCC,ROC_AUC
0,No Label,No Colour + Morphology,37,0,849,121,1.000000,0.875258,0.234177,0.379487,0.452731,0.973363
5,No Label,Colour + No Morphology,37,0,849,121,1.000000,0.875258,0.234177,0.379487,0.452731,0.973697
10,No Label,Colour + Morphology,37,0,848,122,1.000000,0.874227,0.232704,0.377551,0.451039,0.973307
15,No Label,No Colour + No Morphology,37,0,851,119,1.000000,0.877320,0.237179,0.383420,0.456160,0.972889
20,L1,No Colour + Morphology,36,1,854,116,0.972973,0.880412,0.236842,0.380952,0.448463,0.972945
25,L1,Colour + No Morphology,37,0,849,121,1.000000,0.875258,0.234177,0.379487,0.452731,0.973697
30,L1,Colour + Morphology,36,1,854,116,0.972973,0.880412,0.236842,0.380952,0.448463,0.973335
35,L1,No Colour + No Morphology,37,0,851,119,1.000000,0.877320,0.237179,0.383420,0.456160,0.972889
40,L2,No Colour + Morphology,35,2,841,129,0.945946,0.867010,0.213415,0.348259,0.414208,0.969351
45,L2,Colour + No Morphology,37,0,849,121,1.000000,0.875258,0.234177,0.379487,0.452731,0.973697



Metrics for Fold 2


,Label_Set,Strategy,TP,FN,TN,FP,Sensitivity,Specificity,PPV,F1_Score,MCC,ROC_AUC
1,No Label,No Colour + Morphology,32,4,865,106,0.888889,0.890834,0.231884,0.367816,0.420971,0.957146
6,No Label,Colour + No Morphology,33,3,862,109,0.916667,0.887745,0.232394,0.370787,0.429128,0.961752
11,No Label,Colour + Morphology,32,4,863,108,0.888889,0.888774,0.228571,0.363636,0.417329,0.956460
16,No Label,No Colour + No Morphology,32,4,864,107,0.888889,0.889804,0.230216,0.365714,0.419141,0.959149
21,L1,No Colour + Morphology,32,4,872,99,0.888889,0.898043,0.244275,0.383234,0.434321,0.956402
26,L1,Colour + No Morphology,33,3,862,109,0.916667,0.887745,0.232394,0.370787,0.429128,0.961752
31,L1,Colour + Morphology,32,4,868,103,0.888889,0.893924,0.237037,0.374269,0.426573,0.955687
36,L1,No Colour + No Morphology,32,4,864,107,0.888889,0.889804,0.230216,0.365714,0.419141,0.959149
41,L2,No Colour + Morphology,32,4,869,102,0.888889,0.894954,0.238806,0.376471,0.428480,0.952598
46,L2,Colour + No Morphology,33,3,862,109,0.916667,0.887745,0.232394,0.370787,0.429128,0.961752



Metrics for Fold 3


,Label_Set,Strategy,TP,FN,TN,FP,Sensitivity,Specificity,PPV,F1_Score,MCC,ROC_AUC
2,No Label,No Colour + Morphology,33,3,867,104,0.916667,0.892894,0.240876,0.381503,0.438420,0.958863
7,No Label,Colour + No Morphology,33,3,869,102,0.916667,0.894954,0.244444,0.385965,0.442271,0.962467
12,No Label,Colour + Morphology,33,3,870,101,0.916667,0.895984,0.246269,0.388235,0.444227,0.959807
17,No Label,No Colour + No Morphology,33,3,867,104,0.916667,0.892894,0.240876,0.381503,0.438420,0.958977
22,L1,No Colour + Morphology,33,3,868,103,0.916667,0.893924,0.242647,0.383721,0.440336,0.961094
27,L1,Colour + No Morphology,33,3,869,102,0.916667,0.894954,0.244444,0.385965,0.442271,0.962467
32,L1,Colour + Morphology,34,2,872,99,0.944444,0.898043,0.255639,0.402367,0.462001,0.962896
37,L1,No Colour + No Morphology,33,3,867,104,0.916667,0.892894,0.240876,0.381503,0.438420,0.958977
42,L2,No Colour + Morphology,32,4,871,100,0.888889,0.897013,0.242424,0.380952,0.432353,0.956746
47,L2,Colour + No Morphology,33,3,869,102,0.916667,0.894954,0.244444,0.385965,0.442271,0.962467



Metrics for Fold 4


,Label_Set,Strategy,TP,FN,TN,FP,Sensitivity,Specificity,PPV,F1_Score,MCC,ROC_AUC
3,No Label,No Colour + Morphology,33,3,873,98,0.916667,0.899073,0.251908,0.395210,0.450220,0.964441
8,No Label,Colour + No Morphology,32,4,872,99,0.888889,0.898043,0.244275,0.383234,0.434321,0.959149
13,No Label,Colour + Morphology,33,3,872,99,0.916667,0.898043,0.250000,0.392857,0.448201,0.963754
18,No Label,No Colour + No Morphology,33,3,867,104,0.916667,0.892894,0.240876,0.381503,0.438420,0.963554
23,L1,No Colour + Morphology,33,3,874,97,0.916667,0.900103,0.253846,0.397590,0.452261,0.963554
28,L1,Colour + No Morphology,32,4,872,99,0.888889,0.898043,0.244275,0.383234,0.434321,0.959149
33,L1,Colour + Morphology,32,4,872,99,0.888889,0.898043,0.244275,0.383234,0.434321,0.962238
38,L1,No Colour + No Morphology,33,3,867,104,0.916667,0.892894,0.240876,0.381503,0.438420,0.963554
43,L2,No Colour + Morphology,33,3,858,113,0.916667,0.883625,0.226027,0.362637,0.422018,0.959091
48,L2,Colour + No Morphology,32,4,872,99,0.888889,0.898043,0.244275,0.383234,0.434321,0.959149



Metrics for Fold 5


,Label_Set,Strategy,TP,FN,TN,FP,Sensitivity,Specificity,PPV,F1_Score,MCC,ROC_AUC
4,No Label,No Colour + Morphology,31,5,879,92,0.861111,0.905252,0.252033,0.389937,0.434527,0.956917
9,No Label,Colour + No Morphology,31,5,874,97,0.861111,0.900103,0.242188,0.378049,0.424295,0.936349
14,No Label,Colour + Morphology,31,5,880,91,0.861111,0.906282,0.254098,0.392405,0.436644,0.956431
19,No Label,No Colour + No Morphology,31,5,878,93,0.861111,0.904222,0.250000,0.387500,0.432435,0.955859
24,L1,No Colour + Morphology,31,5,880,91,0.861111,0.906282,0.254098,0.392405,0.436644,0.958062
29,L1,Colour + No Morphology,31,5,874,97,0.861111,0.900103,0.242188,0.378049,0.424295,0.936349
34,L1,Colour + Morphology,31,5,880,91,0.861111,0.906282,0.254098,0.392405,0.436644,0.956488
39,L1,No Colour + No Morphology,31,5,878,93,0.861111,0.904222,0.250000,0.387500,0.432435,0.955859
44,L2,No Colour + Morphology,32,4,866,105,0.888889,0.891864,0.233577,0.369942,0.422819,0.951453
49,L2,Colour + No Morphology,31,5,874,97,0.861111,0.900103,0.242188,0.378049,0.424295,0.936349
